# 📄 Hybrid RAG Chatbot
### FAISS IndexFlatIP · Lemmatized BM25 · RRF Fusion · Query Expansion · Embedding Cache · Streaming


## 📦 Cell 1 — Install Dependencies
> Run once. Restart kernel after.

In [1]:
%pip install sentence-transformers faiss-cpu rank-bm25 google-generativeai PyPDF2 nltk
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
print('\n✅ Done. Restart kernel if this was your first install.')

INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this ru

## 🔐 Cell 2 — API Key Setup

**Windows — set permanently (run once in CMD/PowerShell, then restart Jupyter):**
```
setx GEMINI_API_KEY "your_key_here"
```

**Or set just for this session (paste key below):**

In [2]:
import os

# Paste your key here for session-only use
os.environ['GEMINI_API_KEY'] = 'GEMINI_API_KEY'

# Verify
key = os.getenv('GEMINI_API_KEY', '')
if key and key != 'your_key_here':
    print(f'✅ Key loaded: {key[:8]}...{key[-4:]}')
else:
    print('❌ Key not set! Paste your key above.')

✅ Key loaded: GEMINI_A..._KEY


## 📚 Cell 3 — Imports

In [3]:
import os, re, time, json, hashlib, pickle
import numpy as np
import faiss
import PyPDF2
import nltk
import google.generativeai as genai

from pathlib import Path
from functools import lru_cache
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from IPython.display import display, Markdown, clear_output

print('✅ All imports successful')

C:\Users\hp\AppData\Local\Temp\ipykernel_1432\2364613623.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ All imports successful


## ⚙️ Cell 4 — Configuration

In [5]:
PDF_PATH          = 'Rag2.pdf'
from pathlib import Path
CACHE_DIR         = Path('rag_cache')
CACHE_DIR.mkdir(exist_ok=True)

# ── Chunking ──────────────────────────────────────────────────────────────────
CHUNK_SIZE        = 400    # REDUCED from 600 → fewer tokens per chunk
CHUNK_OVERLAP     = 80

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOP_K_RETRIEVE    = 6      # REDUCED from 8 → fewer candidates
TOP_K_RERANK      = 2      # REDUCED from 3 → saves ~260 tokens per query

# ── Token budget ─────────────────────────────────────────────────────────────
# Hard cap on total context chars sent to Gemini.
# 2000 chars ≈ 570 tokens — enough for a complete answer, not wasteful.
# If reranked chunks exceed this, they get truncated before the prompt is built.
MAX_CONTEXT_CHARS = 2000

# ── Sentence window ───────────────────────────────────────────────────────────
# DISABLED by default — was the biggest token culprit.
# 3 reranked chunks × ±1 neighbours = up to 9 chunks = +1850 tokens.
# The retrieval + reranker is already accurate enough without it.
# Set to True only if you notice answers are missing surrounding context.
USE_SENTENCE_WINDOW  = False
WINDOW_SIZE          = 1

# ── Query expansion ───────────────────────────────────────────────────────────
# Each expansion = 1 extra Gemini API call + 3× the retrieval work.
# Only enable when queries are vague or retrieval misses.
USE_QUERY_EXPANSION  = False  # DISABLED by default

# ── Streaming ─────────────────────────────────────────────────────────────────
USE_STREAMING        = True

print('✅ Config set')
print(f'   Chunk size      : {CHUNK_SIZE}  |  Overlap: {CHUNK_OVERLAP}')
print(f'   Retrieve k      : {TOP_K_RETRIEVE}  |  Rerank top: {TOP_K_RERANK}')
print(f'   Max context     : {MAX_CONTEXT_CHARS} chars (~{MAX_CONTEXT_CHARS//4} tokens)')
print(f'   Sentence window : {USE_SENTENCE_WINDOW}')
print(f'   Query expansion : {USE_QUERY_EXPANSION}')

✅ Config set
   Chunk size      : 400  |  Overlap: 80
   Retrieve k      : 6  |  Rerank top: 2
   Max context     : 2000 chars (~500 tokens)
   Sentence window : False
   Query expansion : False


## 🤖 Cell 5 — Load Models
> Run once. Takes ~30 seconds on first run.

In [6]:
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
assert GEMINI_API_KEY, '❌ API key not set. Go back to Cell 2.'
genai.configure(api_key=GEMINI_API_KEY)
print('✅ Gemini configured')

print('Loading embedding model...')
t0 = time.time()
embed_model = SentenceTransformer('multi-qa-mpnet-base-dot-v1')
print(f'✅ Embedding model ready  ({time.time()-t0:.1f}s)')

print('Loading reranker...')
t0 = time.time()
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f'✅ Reranker ready  ({time.time()-t0:.1f}s)')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
print('✅ BM25 lemmatizer ready')

print('\n🎉 All models ready!')

✅ Gemini configured
Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker ready  (6.3s)
✅ BM25 lemmatizer ready

🎉 All models ready!


## 📄 Cell 6 — Load PDF

In [7]:
def load_pdf(path: str) -> str:
    """
    Extract and clean text from PDF.
    Fixes:
      - Hyphenated line breaks: 'intelli-\ngence' → 'intelligence'
      - Normalises all whitespace to single spaces
    """
    text = ''
    with open(path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        print(f'   Pages: {len(reader.pages)}')
        for page in reader.pages:
            page_text = page.extract_text() or ''
            page_text = re.sub(r'-\s*\n\s*', '', page_text)   # fix hyphen breaks
            page_text = re.sub(r'\s+', ' ', page_text)          # normalise spaces
            text += page_text + ' '
    return text.strip()


print(f'Loading {PDF_PATH}...')
document_text = load_pdf(PDF_PATH)

print(f'\n📊 Document Stats:')
print(f'   Characters : {len(document_text):,}')
print(f'   Words      : {len(document_text.split()):,}')
print(f'\nPreview (first 400 chars):')
print('-'*50)
print(document_text[:400])
print('-'*50)

Loading Rag2.pdf...
   Pages: 1

📊 Document Stats:
   Characters : 1,954
   Words      : 329

Preview (first 400 chars):
--------------------------------------------------
Nordlicht Solar GmbH - Employee Handbook (fictional sample) 1. About the company Nordlicht Solar GmbH was founded in 2012 in Cottbus by Anna Keller. The company designs and installs rooftop solar systems for homes and small businesses. It currently employs 85 people. The head office is located at Lausitzer Strasse 14 in Cottbus. 2. Working hours and leave The standard working week is 38 hours. Cor
--------------------------------------------------


## ✂️ Cell 7 — Sentence-Aware Chunking with Overlap

**Why not simple character splitting?**
- Simple: `text[0:400]`, `text[400:800]` → cuts mid-word or mid-sentence
- This version: splits only at `.!?` → every chunk is a complete thought
- Overlap: last 80 chars of chunk N become the start of chunk N+1 → no context lost at boundaries

In [8]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ''

    for sentence in sentences:
        if len(current) + len(sentence) <= chunk_size:
            current += (' ' if current else '') + sentence
        else:
            if current:
                chunks.append(current.strip())
            # carry overlap from end of current chunk into next
            overlap_text = current[-overlap:] if len(current) > overlap else current
            current = overlap_text + ' ' + sentence

    if current.strip():
        chunks.append(current.strip())

    # drop noise chunks (page headers, single words, etc.)
    return [c for c in chunks if len(c) > 50]


chunks = chunk_text(document_text)
lengths = [len(c) for c in chunks]

print(f'📊 Chunking Results:')
print(f'   Total chunks : {len(chunks)}')
print(f'   Min / Avg / Max : {min(lengths)} / {int(sum(lengths)/len(lengths))} / {max(lengths)} chars')
print(f'\nFirst 3 chunks:')
for i, c in enumerate(chunks[:3]):
    print(f'\n--- Chunk {i+1} ({len(c)} chars) ---')
    print(c)

📊 Chunking Results:
   Total chunks : 7
   Min / Avg / Max : 168 / 347 / 396 chars

First 3 chunks:

--- Chunk 1 (396 chars) ---
Nordlicht Solar GmbH - Employee Handbook (fictional sample) 1. About the company Nordlicht Solar GmbH was founded in 2012 in Cottbus by Anna Keller. The company designs and installs rooftop solar systems for homes and small businesses. It currently employs 85 people. The head office is located at Lausitzer Strasse 14 in Cottbus. 2. Working hours and leave The standard working week is 38 hours.

--- Chunk 2 (365 chars) ---
14 in Cottbus. 2. Working hours and leave The standard working week is 38 hours. Core working hours are from 10:00 to 15:00. Every full-time employee receives 28 days of paid vacation per year. Unused vacation days can be carried over until March 31 of the following year. Employees must submit vacation requests to their team lead at least two weeks in advance. 3.

--- Chunk 3 (395 chars) ---
st submit vacation requests to their team lead at 

## 🔍 Cell 8 — Debug: Inspect Chunks (optional)

In [10]:
# Inspect any specific chunk by index
INSPECT_INDEX = 4   # ← change this
print(f'Chunk {INSPECT_INDEX}  ({len(chunks[INSPECT_INDEX])} chars)')
print('-'*50)
print(chunks[INSPECT_INDEX])

# Search for a keyword across all chunks
KEYWORD = 'Anna Keller'   # ← change this
matches = [(i, c) for i, c in enumerate(chunks) if KEYWORD.lower() in c.lower()]
print(f'\n🔎 "{KEYWORD}" found in {len(matches)} chunks')
for idx, chunk in matches[:2]:
    print(f'\n  Chunk {idx}: {chunk[:200]}...')

Chunk 4  (355 chars)
--------------------------------------------------
ass is reimbursed in full. Hotel costs are reimbursed up to 110 euros per night. Meals during business trips are covered with a daily allowance of 28 euros. Expense reports must be submitted within 30 days using form EX-204. 6. IT and security All employees must use two-factor authentication for company accounts. Passwords must be changed every 90 days.

🔎 "Anna Keller" found in 1 chunks

  Chunk 0: Nordlicht Solar GmbH - Employee Handbook (fictional sample) 1. About the company Nordlicht Solar GmbH was founded in 2012 in Cottbus by Anna Keller. The company designs and installs rooftop solar syst...


## 🗂️ Cell 9 — Build FAISS HNSW + BM25 Indexes

| Index | Type | Best for |
|---|---|---|
| FAISS HNSW | Dense / semantic | Meaning-based queries — finds relevant chunks even without exact words |
| BM25 | Sparse / keyword | Exact term matches — names, acronyms, specific phrases |

Using both together (hybrid) gives better coverage than either alone.

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# SPEED IMPROVEMENT 1 — Embedding Cache
#
# Problem: Every time you restart Jupyter and re-run this cell,
#          embed_model.encode(chunks) runs for 20-40 seconds.
#
# Fix: Hash the chunks content. If the hash matches a saved .pkl file,
#      load embeddings from disk (< 1 second) instead of re-encoding.
#      Only re-encodes when the document actually changes.
# ─────────────────────────────────────────────────────────────────────────────

def get_embeddings_cached(chunks: list, model, cache_dir: Path) -> np.ndarray:
    # Hash the chunks content — if document changes, cache is invalidated
    content_hash = hashlib.md5(''.join(chunks).encode()).hexdigest()[:12]
    cache_file   = cache_dir / f'embeddings_{content_hash}.pkl'

    if cache_file.exists():
        print(f'⚡ Loading embeddings from cache ({cache_file.name})...')
        with open(cache_file, 'rb') as f:
            return pickle.load(f)

    print('🔢 Generating embeddings (will cache for next run)...')
    t0  = time.time()
    emb = model.encode(chunks, show_progress_bar=True, normalize_embeddings=True)
    emb = np.array(emb).astype('float32')
    print(f'   Done in {time.time()-t0:.1f}s')

    with open(cache_file, 'wb') as f:
        pickle.dump(emb, f)
    print(f'   Saved to {cache_file.name}')
    return emb


embeddings  = get_embeddings_cached(chunks, embed_model, CACHE_DIR)
print(f'✅ Embeddings shape: {embeddings.shape}')

# ── FAISS IndexFlatIP (exact) ─────────────────────────────────────────────────
dimension   = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)
print(f'✅ FAISS IndexFlatIP  |  {faiss_index.ntotal} vectors')

# ── BM25 with lemmatization ───────────────────────────────────────────────────
def lemmatize_tokens(text: str) -> list:
    tokens = re.findall(r'[a-z]+', text.lower())
    return [lemmatizer.lemmatize(t) for t in tokens
            if t not in stop_words and len(t) > 2]

tokenized_chunks = [lemmatize_tokens(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f'✅ BM25  |  {len(chunks)} documents')

🔢 Generating embeddings (will cache for next run)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   Done in 0.8s
   Saved to embeddings_ef66851f607f.pkl
✅ Embeddings shape: (7, 768)
✅ FAISS IndexFlatIP  |  7 vectors
✅ BM25  |  7 documents


## 🔍 Cell 10 — Retrieval Functions

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# SPEED IMPROVEMENT 2 — Query Embedding Cache
#
# Problem: The same question asked twice re-runs embed_model.encode()
#          each time — 100-200ms wasted.
# Fix: lru_cache stores the last 128 unique query embeddings in memory.
#      Second call with same query = instant, zero model inference.
# ─────────────────────────────────────────────────────────────────────────────

@lru_cache(maxsize=128)
def encode_query(query: str) -> np.ndarray:
    """Encode and cache query embedding. Same query = instant on repeat."""
    return embed_model.encode(
        [query], normalize_embeddings=True
    ).astype('float32')


def vector_search(query: str, k: int = TOP_K_RETRIEVE) -> list:
    """Exact semantic search using cached query embedding."""
    q_emb = encode_query(query)
    scores, indices = faiss_index.search(q_emb, k)
    # Return (chunk, score) pairs so RRF can use the rank positions
    return [(chunks[i], float(scores[0][r]))
            for r, i in enumerate(indices[0]) if i < len(chunks)]


def bm25_search(query: str, k: int = TOP_K_RETRIEVE) -> list:
    """Keyword search with lemmatized tokens."""
    tokens = lemmatize_tokens(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top_idx]


# ─────────────────────────────────────────────────────────────────────────────
# RETRIEVAL IMPROVEMENT 1 — Reciprocal Rank Fusion (RRF)
#
# Problem: Old merging was just order-preserving dedup — it kept the order
#          from vector results and appended new BM25 results at the end.
#          A chunk ranked #1 by BM25 but #6 by vector got pushed to the back.
#
# Fix: RRF scores every chunk based on its RANK in each list, not its raw score.
#      Formula: RRF_score = Σ 1 / (k + rank)   where k=60 (standard constant)
#
# Example:
#   Chunk A: vector rank 3,  BM25 rank 1  → 1/(60+3)  + 1/(60+1)  = 0.032
#   Chunk B: vector rank 1,  BM25 rank 8  → 1/(60+1)  + 1/(60+8)  = 0.031
#   Chunk A wins — consistently good across both retrievers beats one-sided dominance
# ─────────────────────────────────────────────────────────────────────────────

def reciprocal_rank_fusion(result_lists: list, k: int = 60) -> list:
    """
    Merge multiple ranked result lists using RRF.
    result_lists: list of lists of (chunk_text, score) tuples
    Returns: list of chunk_text sorted by fused RRF score (best first)
    """
    rrf_scores = {}
    chunk_map  = {}   # text_key → full chunk text

    for result_list in result_lists:
        for rank, (chunk, _) in enumerate(result_list):
            key = chunk.strip()[:100]
            chunk_map[key] = chunk
            rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (k + rank + 1)

    ranked_keys = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [chunk_map[k] for k in ranked_keys]


# ─────────────────────────────────────────────────────────────────────────────
# RETRIEVAL IMPROVEMENT 2 — Sentence Window Context
#
# Problem: A retrieved chunk might contain the right keywords but the
#          actual answer sentence sits in the chunk just before or after it.
#
# Fix: After reranking, expand each selected chunk to include its
#      neighbouring chunks (±WINDOW_SIZE). Gemini gets broader context
#      without making chunks permanently larger.
# ─────────────────────────────────────────────────────────────────────────────

def expand_with_window(selected_chunks: list, all_chunks: list, window: int = WINDOW_SIZE) -> list:
    """
    For each selected chunk, include surrounding chunks for wider context.
    Deduplicates overlapping windows automatically.
    """
    chunk_index = {c.strip()[:80]: i for i, c in enumerate(all_chunks)}
    included    = set()
    result      = []

    for chunk in selected_chunks:
        idx = chunk_index.get(chunk.strip()[:80])
        if idx is None:
            result.append(chunk)
            continue
        start = max(0, idx - window)
        end   = min(len(all_chunks), idx + window + 1)
        for j in range(start, end):
            if j not in included:
                included.add(j)
                result.append(all_chunks[j])

    return result


# ─────────────────────────────────────────────────────────────────────────────
# SPEED IMPROVEMENT 3 — Reranker Threshold
#
# Problem: CrossEncoder runs inference on every candidate even when
#          one chunk is obviously the best match.
# Fix: If the top vector similarity score is already very high (> threshold),
#      skip the reranker and return immediately. Saves 200-500ms per query.
# ─────────────────────────────────────────────────────────────────────────────

def rerank(query: str, documents: list, top_k: int = TOP_K_RERANK) -> list:
    """Rerank with optional threshold skip."""
    if not documents:
        return []
    scores = reranker.predict([[query, doc] for doc in documents])
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_k]]


def hybrid_search(query: str, debug: bool = False) -> str:
    """Full pipeline: vector + BM25 → RRF fusion → sentence window → rerank."""
    v_results = vector_search(query)
    b_results = bm25_search(query)

    # RRF fusion (replaces old order-preserving dedup)
    fused = reciprocal_rank_fusion([v_results, b_results])

    if debug:
        print(f'Vector : {len(v_results)} | BM25: {len(b_results)} | Fused: {len(fused)}')

    # Rerank the fused list
    top_chunks = rerank(query, fused)

    # Sentence-window expansion
    if USE_SENTENCE_WINDOW:
        top_chunks = expand_with_window(top_chunks, chunks)

    if debug:
        print(f'After window expansion: {len(top_chunks)} chunks')
        for i, c in enumerate(top_chunks):
            print(f'\n  [{i+1}] {c[:250]}...')

    return '\n\n---\n\n'.join(top_chunks)


print('✅ Search functions ready')


✅ Search functions ready


## Cell 11 — Debug: Test Retrieval


In [13]:
test_query = ' I am "FB10330'   # ← change to any topic in your document

print(f'Query: "{test_query}"')
print('='*60)
context = hybrid_search(test_query, debug=True)
print('\n' + '='*60)
print('CONTEXT SENT TO GEMINI:')
print('='*60)
print(context)

Query: " I am "FB10330"
Vector : 6 | BM25: 6 | Fused: 7
After window expansion: 2 chunks

  [1] Nordlicht Solar GmbH - Employee Handbook (fictional sample) 1. About the company Nordlicht Solar GmbH was founded in 2012 in Cottbus by Anna Keller. The company designs and installs rooftop solar systems for homes and small businesses. It currently e...

  [2] ass is reimbursed in full. Hotel costs are reimbursed up to 110 euros per night. Meals during business trips are covered with a daily allowance of 28 euros. Expense reports must be submitted within 30 days using form EX-204. 6. IT and security All em...

CONTEXT SENT TO GEMINI:
Nordlicht Solar GmbH - Employee Handbook (fictional sample) 1. About the company Nordlicht Solar GmbH was founded in 2012 in Cottbus by Anna Keller. The company designs and installs rooftop solar systems for homes and small businesses. It currently employs 85 people. The head office is located at Lausitzer Strasse 14 in Cottbus. 2. Working hours and leave The st

## 🤖 Cell 12 — Gemini Answer Generation

In [17]:
conversation_history = []   # each entry: {'role', 'text', 'summary'}


def summarise_for_history(answer: str) -> str:
    """
    Store a SHORT summary in history instead of the full answer.

    WHY THIS MATTERS FOR TOKENS:
      Old: history stores full Gemini answers (400-800 tokens each)
           After 5 questions: ~2000-4000 tokens just from history
      New: history stores 1-line summaries (~15-20 tokens each)
           After 5 questions: ~75-100 tokens from history
           Savings: ~2000+ tokens per session
    """
    # Take first 2 sentences only — enough to support follow-up questions
    sentences = re.split(r'(?<=[.!?])\s+', answer.strip())
    # Drop the 'Confidence: ...' line from summary
    sentences = [s for s in sentences if not s.lower().startswith('confidence')]
    summary = ' '.join(sentences[:2])
    return summary[:200]  # hard cap at 200 chars


def build_context(chunks_text: str) -> str:
    """
    Apply MAX_CONTEXT_CHARS cap before sending to Gemini.

    WHY THIS MATTERS:
      Sentence window + large chunks can push context to 2000+ tokens.
      This hard cap ensures context never exceeds the budget regardless
      of chunk size, window size, or document verbosity.
    """
    if len(chunks_text) <= MAX_CONTEXT_CHARS:
        return chunks_text
    # Truncate at last complete sentence within the limit
    truncated = chunks_text[:MAX_CONTEXT_CHARS]
    last_period = max(truncated.rfind('.'), truncated.rfind('?'), truncated.rfind('!'))
    if last_period > MAX_CONTEXT_CHARS * 0.7:   # only cut at sentence if not too short
        truncated = truncated[:last_period+1]
    return truncated + '\n[context truncated to stay within token budget]'


def search_with_expansion(question: str, debug: bool = False) -> str:
    """
    Entry point for retrieval.
    USE_QUERY_EXPANSION = True  → runs 3 query variants, fuses with RRF
    USE_QUERY_EXPANSION = False → calls hybrid_search directly (default, cheaper)
    """
    if not USE_QUERY_EXPANSION:
        return hybrid_search(question, debug=debug)   # direct — no extra API call

    # Query expansion: Gemini rewrites question into 3 variants
    try:
        model  = genai.GenerativeModel('gemini-3.8-flash')
        prompt = (
            f'Generate 2 alternative phrasings of this question for document search.\n'
            f'Return ONLY a JSON list of 2 strings. No explanation.\n'
            f'Original: {question}'
        )
        raw      = model.generate_content(prompt).text.strip()
        raw      = re.sub(r'^```[a-z]*\n?|\n?```$', '', raw).strip()
        variants = json.loads(raw)
        queries  = [question] + variants[:2]
    except Exception:
        queries  = [question]   # safe fallback — behaves like expansion=False

    if debug:
        print(f'Query variants: {queries}')

    all_v, all_b = [], []
    for q in queries:
        all_v.extend(vector_search(q))
        all_b.extend(bm25_search(q))

    fused      = reciprocal_rank_fusion([all_v, all_b])
    top_chunks = rerank(question, fused)

    if USE_SENTENCE_WINDOW:
        top_chunks = expand_with_window(top_chunks, chunks)

    return '\n\n---\n\n'.join(top_chunks)


def ask_gemini(question: str, context_raw: str, stream: bool = USE_STREAMING) -> str:
    """
    Generate answer with token-efficient prompt.

    Token optimisations applied:
      1. Context capped at MAX_CONTEXT_CHARS before prompt is built
      2. History uses 1-line summaries not full answers
      3. Only last 2 turns of history included (not 4)
      4. Rules kept concise
    """
    model   = genai.GenerativeModel('gemini-3.8-flash')
    context = build_context(context_raw)

    # History: only last 2 turns (1 exchange), using summaries not full text
    history_text = ''
    if conversation_history:
        recent = conversation_history[-2:]   # CHANGED: was [-4:]
        lines = [
            f"{'User' if t['role']=='user' else 'Bot'}: {t['text']}"
            for t in recent
        ]
        history_text = '\n'.join(lines)

    prompt = f'''Answer the question using ONLY the context below. If not in context say "I don't know from this document."
Be concise. End with: Confidence: High/Medium/Low.
{chr(10)+"Recent: "+history_text+chr(10) if history_text else ""}
Context: {context}

Q: {question}
A:'''

    # Log token estimate before sending
    estimated_tokens = len(prompt) // 4
    print(f'  [~{estimated_tokens} input tokens]')

    if stream:
        response  = model.generate_content(prompt, stream=True)
        full_text = ''
        print('Bot: ', end='', flush=True)
        for chunk in response:
            print(chunk.text, end='', flush=True)
            full_text += chunk.text
        print()
        return full_text.strip()
    else:
        response = model.generate_content(prompt)
        return response.text.strip()


# Quick test
q = '"How many vacation days do employees get?"'
print(f'Q: {q}')
a = ask_gemini(q, hybrid_search(q))
print(f'\nAnswer length: {len(a)} chars')

Q: "How many vacation days do employees get?"
  [~246 input tokens]
Bot: Full-time employees receive 28 days of paid vacation per year.

Confidence: High

Answer length: 80 chars


## 💬 Cell 13 — Chat
> Type your question and press Enter. Type `exit` to stop.

In [ ]:
def chat():
    global conversation_history
    conversation_history = []

    print('='*50)
    print('  RAG Chatbot — token-efficient mode')
    print("  Type 'debug' before question for retrieval details")
    print("  Type 'tokens' to see token usage estimate")
    print("  Type 'exit' to stop")
    print('='*50 + '\n')

    total_tokens_est = 0

    while True:
        query = input('You: ').strip()
        if not query:
            continue
        if query.lower() in ('exit', 'quit', 'bye'):
            print(f'Bot: Goodbye!  Session used ~{total_tokens_est:,} input tokens.')
            break
        if query.lower() == 'tokens':
            print(f'  Estimated input tokens this session: ~{total_tokens_est:,}')
            print(f'  History entries: {len(conversation_history)}')
            continue

        debug = False
        if query.lower().startswith('debug '):
            debug = True
            query = query[6:].strip()

        t0      = time.time()
        context = search_with_expansion(query, debug=debug)
        answer  = ask_gemini(query, context)
        t_total = time.time() - t0

        total_tokens_est += len(context) // 4 + len(query) // 4

        print(f'  ⏱  {t_total:.2f}s\n')

        # Store summary (not full answer) to keep history tokens low
        summary = summarise_for_history(answer)
        conversation_history.append({'role': 'user', 'text': query})
        conversation_history.append({'role': 'bot',  'text': summary})  # ← summary
        if len(conversation_history) > 6:   # keep last 3 exchanges max
            conversation_history = conversation_history[-6:]


chat()

  RAG Chatbot — token-efficient mode
  Type 'debug' before question for retrieval details
  Type 'tokens' to see token usage estimate
  Type 'exit' to stop



You:  Who founded the company and when?


  [~251 input tokens]
Bot: Anna Keller founded the company in 2012.

Confidence: High
  ⏱  10.43s

